Back propagation doesn't make your network works magically. Great things doesn't works automatically. So we need to dive into the real occean of `back propagation`. 


In [1]:
import os 
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
data = "datas\\names.txt" 
data_path = os.path.join(os.getcwd(),data)

In [3]:
text_data = open(data_path,"r").read().splitlines()
type(text_data)

list

In [4]:
chars   = sorted(list(set("".join(text_data))))
stoi    = {s:i+1 for i,s in enumerate(chars)} 
stoi["."] = 0
itos    = {s:i for i,s in sorted((stoi.items()))}

## Building Dataset

In [5]:
def build_dataset(dataset:list=None,block_size:int=3):

    X,Y = [],[]
    for word in dataset:

        context = [0] * block_size
        for char in word +".":
            idx = stoi[char]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx] # crop and append
    
    X = torch.tensor(X,device="cuda")
    Y = torch.tensor(Y,device="cuda") 
    print(X.shape,Y.shape)
    return X,Y


random.seed(42)
random.shuffle(text_data,random=random.seed(42))
n1 = int(0.8 * len(text_data))
n2 = int(0.9 * len(text_data))

x_train,y_train = build_dataset(text_data[:n1])
x_val,y_val     = build_dataset(text_data[n1:n2])
x_test,y_test   = build_dataset(text_data[n2:]) 


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


## Utility Function

In [6]:
## utility function will be use later when comparing manual gradient to pytorch gradient
def compare(s,dt,t):
    ex          = torch.all(dt == t.grad).item()    # dt= manual grad, t= grad calculated by torch
    app         = torch.allclose(dt,t.grad)
    max_diff    = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {max_diff}')



def compare(parameter,manual_grad,torch_grad):
    ex          = torch.all(manual_grad == torch_grad.grad).item()    # dt= manual grad, t= grad calculated by torch
    app         = torch.allclose(manual_grad,torch_grad.grad)
    max_diff    = (manual_grad - torch_grad.grad).abs().max().item()
    print(f'{parameter:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {max_diff}')

In [7]:
len(itos)

27

In [8]:
n_embd      = 10 # the dimensionality of the character embedding vectors
n_hidden    = 64 # the number of neurons in the hidden layer of the MLP
block_size  = 3
vocab_size  = len(itos)
batch_size  = 32 


g   = torch.Generator(device="cuda:0").manual_seed(2147483647) # for reproducibility
C   = torch.randn((vocab_size, n_embd),            generator=g,device="cuda:0")
# Layer 1
w1 = torch.randn((n_embd * block_size, n_hidden),   generator=g,device="cuda:0") * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                          generator=g,device="cuda:0") * 0.1 # using b1 for nothing, it's useless because of BN
# Layer 2
w2 = torch.randn((n_hidden, vocab_size),            generator=g,device="cuda:0") * 0.1 + 1.0
b2 = torch.randn(vocab_size,                        generator=g,device="cuda:0") * 0.1
# BatchNorm parameters 
bn_gain     = torch.randn((1,n_hidden),device="cuda:0") * 0.1 + 1.0
bn_bias     = torch.randn((1,n_hidden),device="cuda:0") * 0.1 


parameters = [C,w1,b1,w2,b2,bn_gain,bn_bias]
print(f"total numbers of parameters in the model    : {sum(para.nelement() for para in parameters)}")
for para in parameters:
    para.requires_grad = True

total numbers of parameters in the model    : 4137


- Above i set `bias` to be not equal to zero, because 
    - If your variables are set to zero, sometimes mask an incorrect implementation of the gradient. 
- `b1` is activated here, but there is no usage, because it cancelled out by `batch_normalization`. 


## Mini batch 

In [9]:
batch_size      = 32
n               = batch_size # for convenience

idx             = torch.randint(0,x_train.shape[0],(batch_size,),generator=g,device="cuda:0")
x_batch,y_batch = x_train[idx],y_train[idx]

In [10]:
emb         = C[x_batch]                                # (32,3,10)         ==>> (batch_size, block_size, n_embd)
emb_cat     = emb.view((emb.shape[0]),-1)               # (32,30)           ==>> (batch_size, (block_size,n_emb))
# -----------------------------------------------------------------------------------------------------------------------
#  linear layer 
h_preact_n  = emb_cat @ w1 + b1                         #(32 , 30) * (30,64) ==>> (32,64)   


In [11]:
emb         = C[x_batch]                                # (32,3,10)         ==>> (batch_size, block_size, n_embd)
emb_cat     = emb.view((emb.shape[0]),-1)               # (32,30)           ==>> (batch_size, (block_size,n_emb))
# -----------------------------------------------------------------------------------------------------------------------
#  linear layer 
h_preact_n  = emb_cat @ w1 + b1                         #(32 , 30) * (30,64) ==>> (32,64)   
# ------------------------------------------------------------------------------------------------------------------------
# Batch_normalization
bn_mean_i   = (1/n) * h_preact_n.sum(0,keepdim=True)      # (1,64)
bn_diff     = h_preact_n - bn_mean_i                      # (32,64)
bn_diff_sqr = bn_diff ** 2 
bn_var      = 1 / (n-1) * (bn_diff_sqr).sum(0,keepdim=True) # (1,64) note: bessal's correlation (divided by n-1, not by n )
bn_var_invs = (bn_var + 1e-5) ** -0.5
bn_raw      = bn_diff * bn_var_invs
h_preact    = bn_gain * bn_raw + bn_bias
# -------------------------------------------------------------------------------------------------------------------------
# Non linearity 
h = torch.tanh(h_preact)
# -------------------------------------------------------------------------------------------------------------------------
# layer 2 
logits = h_preact @ w2 + b2                     # (32,27)       ==>> (batch_size,vocal_size)
# -------------------------------------------------------------------------------------------------------------------------
# cross entropy 
logits_max  = logits.max(1,keepdim=True)[0]     # (32,1)
norm_logits = logits - logits_max               # (32,27)          for numerical stability (in torch it subtact the max value from tensor avoid errors in calculations)
    # softax
counts          = norm_logits.exp()             # (32,27)
counts_sum      = counts.sum(1,keepdim=True)    # 
counts_sum_invs = counts_sum**-1                # (32,1)            if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs           = counts * counts_sum_invs      # (32,27) * (32,1)  ==>> (32,27)
log_probs       = probs.log()                   # (32,27)
loss            = -log_probs[range(n),y_batch].mean()   
# this loss is same as F.cross_entropy(logits,y_batch)

para_list = [log_probs,probs,counts,counts_sum,counts_sum_invs,norm_logits,logits_max,
          logits,h,h_preact,bn_raw,bn_var_invs,bn_var,bn_diff_sqr,bn_diff,h_preact_n,
          bn_mean_i,emb_cat,emb]

for para in parameters:
    para.grad = None

for t in para_list:
    t.retain_grad()

loss.backward()
loss

tensor(3.4909, device='cuda:0', grad_fn=<NegBackward0>)

## Exercise_1 :



Start implementing backward pass. 
## 1. `log_probs` -->> gradient is -->> `d_log_probs`
    - Calculate the gradient of loss with respect to all the elements of `log_probs`. 
    - In other words, `derivative of loss with respect to all the elemets of log_probs`

- ### loss = -log_probs[range(n),y_batch].mean()   

    - for example,  a =  log_probs[0,y_batch[0]]
    - 
    - loss = - (a + b + c + ........+ n ).mean()  
    - loss = - (a + b + c + ..... + n) / n         ==>> where n, batch_size
    - loss = - (1/n) (a + b + c + .... + n )
    - d(loss) / da = - 1 / n 
    - d(loss) / db = - 1 / n 
    - d(loss) / dc = - 1 / n 
    - d(loss) / dn = - 1 / n

- log_probs.shape  = (32,27), but loss only have only 32. that means in loss calculation only 32 are participating. 
- So what is the derivative of most of the other elements that do not plucked out is zero. 
    - That means, if we change the other elements (not participants in the loss calculations) it doesn't impact on the loss values. 

##------------------------------------------------------------------------------------------## 
- Simple explanation of derivative of loss with respect to elements. 
- consider loss     = (a + b + c) / 3
    - d(loss) / da  = d/da (a + b + c) /3
    - d(loss) / da  = 1/3 * d /da (a + b +c)
        - The derivative of a with respect to aa is 1.
        - The derivative of b with respect to a is 0 (since bb is constant with respect to a).
        - The derivative of c with respect to a is 0 (since cc is constant with respect to a). 

    - d(loss) / db  = d/db (a + b + c) /3
    - d(loss) / db  = 1/3 * d /db (a + b +c)
        - The derivative of a with respect to b is 1.
        - The derivative of b with respect to b is 0 (since b is constant with respect to b).
        - The derivative of c with respect to b is 0 (since c is constant with respect to b).

    - d(loss) / dc  = d/dc (a + b + c) /3
    - d(loss) / dc  = 1/3 * d /dc (a + b +c)
        - The derivative of a with respect to c is 1.
        - The derivative of b with respect to c is 0 (since b is constant with respect to c).
        - The derivative of c with respect to c is 0 (since c is constant with respect to c).  

=========================================================================================================

- Shape of **logprobs is (32,27)**, but only 32 of them are participating in the loss calculations.
    - ie, d(logprobs) / n = -1 / n, where n  = 32 

In [21]:
d_log_probs = torch.zeros_like(log_probs)       # same as torch.zeros((32,27))
d_log_probs[range(n),y_batch] = - 1.0 / n


## compare the manual grad and torch_grad
compare("log_probs",d_log_probs,log_probs)

log_probs       | exact: True  | approximate: True  | maxdiff: 0.0


## 2. `probs` -->> gradient is -->> `d_probs`

- ### log_probs  = probs.log(), 
    - so log_probs depending on `probs`through a log(). so all the elements of probs are being element wise applied  log(). 
    - `probs.log()` mean passing probs into log(), 
        - log(probs) == we can write as log(x)
    - d(loss) / log_probs = d(log(x)) / d(x)
    - d(loss) / log_probs = 1 / x * log_probs
        - where x = probs
    - d(loss) / log_probs = 1 / probs * log_probs

In [12]:
d_probs = (1.0 / probs) * d_log_probs 

compare("probs",d_probs,probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


## 3. `count_sum_invs` -->> gradient is -->> `d_count_sum_invs`

- ### probs = counts * counts_sum_invs

- counts.shape = (32,27) & counts_sum_invs.shape = (32,1)
- c = a * b, 
    - consider a = [3,3], b = [3,1] ==>> [3,3] * [3,1] ==>> (3,3)

Consider the matrices:


A =
\begin{pmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33} \\
\end{pmatrix}
B =
\begin{pmatrix}
b_1 \\
b_2 \\
b_3 \\
\end{pmatrix}

Multiplication of these matrices:

C = A * B 
\begin{pmatrix}
a_{11} \times b_1 & a_{12} \times b_1 & a_{13} \times b_1 \\
a_{21} \times b_2 & a_{22} \times b_2 & a_{23} \times b_2 \\
a_{31} \times b_3 & a_{32} \times b_3 & a_{33} \times b_3 \\
\end{pmatrix}

In [14]:
counts.shape,counts_sum_invs.shape

(torch.Size([32, 27]), torch.Size([32, 1]))

In [16]:
(counts * counts_sum_invs).shape

torch.Size([32, 27])

: 